# Import Library

In [ ]:
import re
import jieba
import pandas as pd
import emoji
from textblob import TextBlob

from transformers import pipeline

# from pysentimiento import create_analyzer
from tqdm import tqdm  # untuk progress bar (optional)

print("Selesaii....")


# Function

In [2]:
def split_hin(row):
    """
    Memisahkan kalimat dari kolom 'Feedback' (berdasarkan titik)
    dan kolom 'Hindi' (berdasarkan koma) untuk satu baris DataFrame.

    Parameters:
    - row (pd.Series): satu baris dari DataFrame

    Returns:
    - List[Dict]: list baris hasil pemisahan
    """
    sentencesE = re.split(r'[.]', str(row['Feedback']))
    sentencesH = re.split(r'[।]', str(row['FeedbackH']))
    
    max_len = max(len(sentencesE), len(sentencesH))
    expanded = []
    for i in range(max_len):
        expanded.append({
            "No": row.get("No"),
            "courseID": row.get("courseID"),
            "user": row.get("User"),
            "rating": row.get("Rating"),
            "time": row.get("Time (bulan)"),
            "feedbackE": sentencesE[i] + '.' if i < len(sentencesE) else '',
            "feedbackH": sentencesH[i] if i < len(sentencesH) else ''
        })
    return expanded

import pandas as pd

def merge_by_no(df):
    agg_rules = {
    "courseID"     : "first",                        # karena pasti sama di dalam grup
    "user"         : lambda x: ", ".join(sorted(set(x))),
    "rating"       : "mean",
    "time"         : "mean",
    "feedback"      : lambda x: " . ".join(x),        # gabungkan semua review
    "aspect"       : list,                           # simpan sebagai list mentah
    # untuk kolom emosi → apakah ada emosi ‘1’ di salah satu baris
    "anger"        : "max",
    "anticipation" : "max",
    "disgust"      : "max",
    "fear"         : "max",
    "joy"          : "max",
    "sadness"      : "max",
    "surprise"     : "max",
    "trust"        : "max",
}

    # ---------- 3) GROUPBY + AGG ----------
    df_final = (
        df
        .groupby("No", as_index=False)
        .agg(agg_rules)
        # opsional: rapikan urutan kolom
        .reindex(columns = ["No", "courseID", "user", "rating","time","feedback",
                            "aspect","anger","anticipation","disgust","fear",
                            "joy","sadness","surprise","trust"])
    )
    return df_final

def count_words_mixed(text: str) -> int:
    """
    Mengembalikan jumlah 'kata' dalam sebuah kalimat:
      • Bila ada karakter Hanzi (范围 U+4E00–U+9FFF) → pakai jieba
      • Selain itu (alfabet / pinyin / latin dsb.)  → split spasi
    """
    if not isinstance(text, str):
        return 0                     # NaN, None, dsb.
    
    # Cek apakah ada karakter Tionghoa
    contains_hanzi = re.search(r'[\u4e00-\u9fff]', text)
    
    if contains_hanzi:
        tokens = jieba.lcut(text)    # ['我', '喜欢', 'Python', '编程']
        # buang token kosong/bias
        tokens = [t for t in tokens if t.strip()]
        return len(tokens)
    
    # Untuk teks biasa (Inggris/Indonesia dsb.)
    return len(text.split())

def find_emoji(text: str):
    """
    Pisahkan emoji dan kembalikan:
    - teks tanpa emoji (atau 0 jika kosong)
    - list emoji yang ditemukan
    """
    
    if not isinstance(text, str):
        return 0, [], 0

    # ambil semua emoji (urut kemunculan)
    emjs = [d['emoji'] for d in emoji.emoji_list(text)]

    # hapus emoji dari teks
    for e in emjs:
        text = text.replace(e, '')
    text_no_emo = text.strip()

    # jika kosong, ganti dengan angka 0
    if text_no_emo == '':
        text_no_emo = 0

    return text_no_emo, emjs, len(emjs)       # kembalikan list emoji

print("Selesai...")

Selesai...


# Extract Emoji

In [ ]:
import pandas as pd

# Read dataset
df = pd.read_excel("dataset/4. newlabel/extend/6indo.xlsx")
# print(df)

# extract emoji
df[['feed_no_emo', '_emo', 'n_emo']] = df['feedback'].apply(lambda t: pd.Series(find_emoji(t)))
df.to_csv("dataset/4. newlabel/extend/6indo.csv", index=False)

In [ ]:
count = df[df['n_emo'] > 0].shape[0]
print(f"Jumlah feedback dengan emojis: {count}, {round((count/len(df))*100, 2)}%")

In [ ]:
# Counting the words
import re

df["word_length"] = df["feedback"].apply(
    lambda x: len(re.findall(r"[A-Za-z']+", x))
)

min_len = df["word_length"].min()
max_len = df["word_length"].max()
avg_len = df["word_length"].mean()

print(f"Min: {min_len}, Max: {max_len}, Avg: {avg_len}")

In [ ]:
# Counting mixing words
df['word_count'] = df['feedback'].fillna('').apply(count_words_mixed)

min_len = df["word_count"].min()
max_len = df["word_count"].max()
avg_len = df["word_count"].mean()

print(f"Min: {min_len}, Max: {max_len}, Avg: {avg_len}")

In [ ]:
# Distribusi Aspect
aspect_count = df["Aspect"].value_counts().sort_index()
print(aspect_count)
print()

sentiment_count = df["Sentiment"].value_counts().sort_index()
print(sentiment_count)
print()

anger_count = df["Anger"].value_counts().sort_index()
print(anger_count)
print()

antcp_count = df["Anticipation"].value_counts().sort_index()
print(antcp_count)
print()

disgust_count = df["Disgust"].value_counts().sort_index()
print(disgust_count)
print()

fear_count = df["Fear"].value_counts().sort_index()
print(fear_count)
print()

joy_count = df["Joy"].value_counts().sort_index()
print(joy_count)
print()

sad_count = df["Sadness"].value_counts().sort_index()
print(sad_count)
print()

surprise_count = df["Surprise"].value_counts().sort_index()
print(surprise_count)
print()

trust_count = df["Trust"].value_counts().sort_index()
print(trust_count)
print()